<a href="https://colab.research.google.com/github/ramossaucedo1980-stack/data-ai-portfolio/blob/main/01_bronze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Paso 1 · Preparar PySpark
Colab ya trae Java **instalado**, así que solo hay que preparar PySpark.

Colab trae también una librería llamada dataproc-spark-connect que choca con PySpark y provoca errores al crear la sesión. Por eso primero la desinstalamos junto con el PySpark que viene de fábrica, y luego instalamos una versión limpia: PySpark 3.5.1.

Si Colab te muestra un aviso de "Restart session", acéptalo y vuelve a ejecutar desde esta celda.**bold text**

In [2]:
get_ipython().system('pip uninstall -y dataproc-spark-connect pyspark')
get_ipython().system('pip install pyspark==3.5.1 -q')

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("medallion").getOrCreate()
print(spark.version)

Found existing installation: pyspark 3.5.1
Uninstalling pyspark-3.5.1:
  Successfully uninstalled pyspark-3.5.1
3.5.1


Paso 2 · Crear la sesión de Spark
La SparkSession es la puerta de entrada a Spark: todo lo que hagas pasa por ella.

appName: nombre de tu aplicación (aparece en logs).
master("local[*]"): corre Spark en esta máquina usando todos los núcleos disponibles.
Deberías ver: Spark 3.5.1 listo.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("medallion")
    .master("local[*]")
    .getOrCreate()
)

print("Spark", spark.version, "listo")

Spark 3.5.1 listo


Paso 3 · Simular la llegada de un archivo CSV "sucio"
En la vida real los datos llegan de sistemas externos y siempre traen errores. Aquí creamos un CSV con 8 ventas y metemos estos errores a propósito:

Venta	Error plantado	Lo arreglamos en
3	Tienda con espacios de más: " Sur "	Silver
4	Precio negativo: -4500	Silver
5	Fecha en otro formato: 02/09/2026	Silver
6	Cliente vacío	Silver
7	Cantidad escrita en texto: dos	Silver
2 (repetida)	Registro duplicado	Silver
El archivo se guarda en /content/raw/ventas_raw.csv (el disco temporal de Colab).

In [4]:
import os

os.makedirs("/content/raw", exist_ok=True)

csv_crudo = """id_venta,tienda,producto,cantidad,precio,fecha,cliente
1,Centro,Laptop,1,15000,2026-09-01,Ana
2,Norte,Mouse,3,250,2026-09-01,Luis
3,  Sur ,Teclado,2,800,2026-09-02,Carla
4,Centro,Monitor,1,-4500,2026-09-02,Pedro
5,Norte,Laptop,2,15000,02/09/2026,Sofia
6,Sur,Mouse,1,250,2026-09-03,
7,Centro,Audífonos,dos,600,2026-09-03,Marta
2,Norte,Mouse,3,250,2026-09-01,Luis
"""

ruta_raw = "/content/raw/ventas_raw.csv"
with open(ruta_raw, "w", encoding="utf-8") as archivo:
    archivo.write(csv_crudo)

print("Archivo creado en:", ruta_raw)
print(open(ruta_raw, encoding="utf-8").read())

Archivo creado en: /content/raw/ventas_raw.csv
id_venta,tienda,producto,cantidad,precio,fecha,cliente
1,Centro,Laptop,1,15000,2026-09-01,Ana
2,Norte,Mouse,3,250,2026-09-01,Luis
3,  Sur ,Teclado,2,800,2026-09-02,Carla
4,Centro,Monitor,1,-4500,2026-09-02,Pedro
5,Norte,Laptop,2,15000,02/09/2026,Sofia
6,Sur,Mouse,1,250,2026-09-03,
7,Centro,Audífonos,dos,600,2026-09-03,Marta
2,Norte,Mouse,3,250,2026-09-01,Luis



Paso 4 · Leer el CSV con Spark (ingesta)
Opciones importantes: - header=True: la primera fila son los nombres de las columnas. - inferSchema=False: todas las columnas se leen como texto (string).

¿Por qué todo como texto? Porque en Bronze no queremos que Spark "adivine" tipos ni descarte valores raros como dos o 02/09/2026. Guardamos exactamente lo que llegó. La conversión de tipos se hace en Silver.

Deberías ver 8 filas y un esquema donde todo dice string.

In [5]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("encoding", "UTF-8")
    .csv(ruta_raw)
)

df_raw.printSchema()
df_raw.show(truncate=False)
print("Filas leídas:", df_raw.count())

root
 |-- id_venta: string (nullable = true)
 |-- tienda: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- cliente: string (nullable = true)

+--------+------+---------+--------+------+----------+-------+
|id_venta|tienda|producto |cantidad|precio|fecha     |cliente|
+--------+------+---------+--------+------+----------+-------+
|1       |Centro|Laptop   |1       |15000 |2026-09-01|Ana    |
|2       |Norte |Mouse    |3       |250   |2026-09-01|Luis   |
|3       |  Sur |Teclado  |2       |800   |2026-09-02|Carla  |
|4       |Centro|Monitor  |1       |-4500 |2026-09-02|Pedro  |
|5       |Norte |Laptop   |2       |15000 |02/09/2026|Sofia  |
|6       |Sur   |Mouse    |1       |250   |2026-09-03|NULL   |
|7       |Centro|Audífonos|dos     |600   |2026-09-03|Marta  |
|2       |Norte |Mouse    |3       |250   |2026-09-01|Luis   |
+--------+------+---------+--

Paso 5 · Agregar columnas de auditoría
Bronze no corrige datos, pero sí registra de dónde y cuándo llegaron. Esto es clave cuando algo falla y necesitas rastrear el origen:

fecha_ingesta: momento exacto en que cargamos el archivo.
fuente: nombre del archivo de origen.

In [6]:
df_bronze = (
    df_raw
    .withColumn("fecha_ingesta", F.current_timestamp())
    .withColumn("fuente", F.lit("ventas_raw.csv"))
)

df_bronze.show(truncate=False)

+--------+------+---------+--------+------+----------+-------+--------------------------+--------------+
|id_venta|tienda|producto |cantidad|precio|fecha     |cliente|fecha_ingesta             |fuente        |
+--------+------+---------+--------+------+----------+-------+--------------------------+--------------+
|1       |Centro|Laptop   |1       |15000 |2026-09-01|Ana    |2026-09-30 15:14:11.793942|ventas_raw.csv|
|2       |Norte |Mouse    |3       |250   |2026-09-01|Luis   |2026-09-30 15:14:11.793942|ventas_raw.csv|
|3       |  Sur |Teclado  |2       |800   |2026-09-02|Carla  |2026-09-30 15:14:11.793942|ventas_raw.csv|
|4       |Centro|Monitor  |1       |-4500 |2026-09-02|Pedro  |2026-09-30 15:14:11.793942|ventas_raw.csv|
|5       |Norte |Laptop   |2       |15000 |02/09/2026|Sofia  |2026-09-30 15:14:11.793942|ventas_raw.csv|
|6       |Sur   |Mouse    |1       |250   |2026-09-03|NULL   |2026-09-30 15:14:11.793942|ventas_raw.csv|
|7       |Centro|Audífonos|dos     |600   |2026-09-03|M

Paso 6 · Guardar la capa Bronze en Parquet
Parquet es el formato estándar en data engineering: guarda por columnas, comprime muy bien y conserva el esquema. Es lo que usan por debajo Iceberg, Delta Lake y la mayoría de los data lakes.

mode("overwrite"): si ya existe, lo reemplaza. Así puedes volver a ejecutar el notebook sin errores.

In [7]:
ruta_bronze = "/content/bronze/ventas"

df_bronze.write.mode("overwrite").parquet(ruta_bronze)

print("Bronze guardado en:", ruta_bronze)
!ls -la /content/bronze/ventas

Bronze guardado en: /content/bronze/ventas
total 20
drwxr-xr-x 2 root root 4096 Sep 30 15:15 .
drwxr-xr-x 3 root root 4096 Sep 30 15:15 ..
-rw-r--r-- 1 root root 2947 Sep 30 15:15 part-00000-a538bba6-b27d-4bec-a27a-f7c097996538-c000.snappy.parquet
-rw-r--r-- 1 root root   32 Sep 30 15:15 .part-00000-a538bba6-b27d-4bec-a27a-f7c097996538-c000.snappy.parquet.crc
-rw-r--r-- 1 root root    0 Sep 30 15:15 _SUCCESS
-rw-r--r-- 1 root root    8 Sep 30 15:15 ._SUCCESS.crc


Paso 7 · Verificar lo que guardamos
Nunca confíes en que se guardó bien: léelo de vuelta. Deberías ver 8 filas y 9 columnas (7 originales + 2 de auditoría).

In [8]:
df_check = spark.read.parquet(ruta_bronze)

print("Filas en Bronze:", df_check.count())
print("Columnas:", len(df_check.columns))
df_check.printSchema()
df_check.show(truncate=False)

Filas en Bronze: 8
Columnas: 9
root
 |-- id_venta: string (nullable = true)
 |-- tienda: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- fecha_ingesta: timestamp (nullable = true)
 |-- fuente: string (nullable = true)

+--------+------+---------+--------+------+----------+-------+--------------------------+--------------+
|id_venta|tienda|producto |cantidad|precio|fecha     |cliente|fecha_ingesta             |fuente        |
+--------+------+---------+--------+------+----------+-------+--------------------------+--------------+
|1       |Centro|Laptop   |1       |15000 |2026-09-01|Ana    |2026-09-30 15:15:52.222702|ventas_raw.csv|
|2       |Norte |Mouse    |3       |250   |2026-09-01|Luis   |2026-09-30 15:15:52.222702|ventas_raw.csv|
|3       |  Sur |Teclado  |2       |800   |2026-09-02|Carla  |2026-09-30 15:15:5

Paso 8 · Detectar los problemas (preparación para Silver)
Aquí no corregimos nada, solo medimos. Es el primer paso de cualquier limpieza: saber qué está mal.

cast("double") y cast("int") intentan convertir texto a número; si no pueden (como con dos), devuelven null. Así detectamos valores inválidos.

In [9]:
print("1) Ventas duplicadas (mismo id_venta):")
df_check.groupBy("id_venta").count().filter(F.col("count") > 1).show()

print("2) Precios negativos:")
df_check.filter(F.col("precio").cast("double") < 0).show(truncate=False)

print("3) Cantidades que no son número:")
df_check.filter(F.col("cantidad").cast("int").isNull()).show(truncate=False)

print("4) Clientes vacíos:")
df_check.filter(F.col("cliente").isNull()).show(truncate=False)

print("5) Tiendas con espacios de más:")
df_check.filter(F.col("tienda") != F.trim(F.col("tienda"))).show(truncate=False)

print("6) Fechas que no tienen formato AAAA-MM-DD:")
df_check.filter(~F.col("fecha").rlike(r"^\d{4}-\d{2}-\d{2}$")).show(truncate=False)

1) Ventas duplicadas (mismo id_venta):
+--------+-----+
|id_venta|count|
+--------+-----+
|       2|    2|
+--------+-----+

2) Precios negativos:
+--------+------+--------+--------+------+----------+-------+--------------------------+--------------+
|id_venta|tienda|producto|cantidad|precio|fecha     |cliente|fecha_ingesta             |fuente        |
+--------+------+--------+--------+------+----------+-------+--------------------------+--------------+
|4       |Centro|Monitor |1       |-4500 |2026-09-02|Pedro  |2026-09-30 15:15:52.222702|ventas_raw.csv|
+--------+------+--------+--------+------+----------+-------+--------------------------+--------------+

3) Cantidades que no son número:
+--------+------+---------+--------+------+----------+-------+--------------------------+--------------+
|id_venta|tienda|producto |cantidad|precio|fecha     |cliente|fecha_ingesta             |fuente        |
+--------+------+---------+--------+------+----------+-------+--------------------------+